# Dự án 2 — Máy đoán nhãn bằng hàng xóm — Đề A: Khối u lành hay ác (đề có hướng dẫn)
## BẢN CỦA NHÓM

**Trường THCS và THPT Đinh Thiện Lý — Machine Learning Level 1 — Khối 10**

---

Ảnh tế bào lấy từ mẫu sinh thiết được đo thành 30 con số. Bộ **Breast Cancer Wisconsin** có 569 mẫu đã có chẩn đoán của bác sĩ. Nhóm thử xem KNN học được tới đâu.
Đề này nối tiếp Bài 14: notebook có nhiều khung hơn.

**Câu hỏi lớn của dự án:** *KNN đoán đúng tới đâu so với một mốc đơn giản — và K nào là tốt?*

**Dữ liệu:** scikit-learn `load_breast_cancer` — Breast Cancer Wisconsin (UCI), CC BY 4.0  

> Đây là bài tập học máy, **không dùng để chẩn đoán** cho bất kỳ ai. Chẩn đoán là việc của bác sĩ.

### Các mốc (nộp trên Canvas trước 23:59 Thứ 7 của tuần đó)

| Tuần | Mốc | Nộp |
|---|---|---|
| 1 · buổi 11 | **M1 + M2** Chuẩn bị dữ liệu + KNN đầu tiên so với mốc | Notebook bước 1 – 3 |
| 2 · buổi 12 | **M3 + M4** Chọn K, chấm test một lần, 1 slide | Notebook đủ bước 1 – 6 + slide + Nhật ký AI |

### Nhóm và vai — **đổi vai mỗi tuần**

Nhóm: … · Lớp: …

| Vai | Tuần 1 | Tuần 2 | Tuần 3 | Việc chính |
|---|---|---|---|---|
| Lái code | … | … | … | Gõ và chạy notebook; không ai gõ hộ hai tuần liền |
| Nghiên cứu | … | … | … | Đọc bài thư viện, hỏi Gemini “vì sao”, ghi Nhật ký AI |
| Thư ký số liệu | … | … | … | Ghi bảng thí nghiệm, kiểm tra con số trước khi báo cáo |
| Trình bày | … | … | … | Báo cáo 1 phút mỗi tuần, làm slide / poster |

### Quy tắc dùng AI

- **Được:** hỏi Gemini (✨ trong Colab) một lệnh làm gì, bấm *Giải thích lỗi*, hỏi *vì sao* làm bước này, nhờ gợi ý cách tiếp cận.
- **Không được:** nộp con số, biểu đồ, kết luận mà nhóm không tự chạy ra; để AI viết cả notebook.
- Ô code do AI gợi ý ghi dòng đầu `# AI gợi ý — nhóm đã kiểm tra: ...`. Mỗi tuần ghi **Nhật ký AI** ở cuối.
- Khi đi qua nhóm và ở mốc cuối, thầy gọi **một bạn bất kỳ** giải thích một dòng code hoặc một con số.

### Dữ liệu

| Cột | Ý nghĩa |
|---|---|
| `mean radius, mean texture, …` | 30 số đo của nhân tế bào trong ảnh (bán kính, kết cấu, diện tích…) |
| `chan_doan` | Nhãn: lành tính / ác tính (do bác sĩ xác định) |


---
# TUẦN 1 (buổi 11) — Chuẩn bị dữ liệu, KNN đầu tiên

**Mốc của tuần:** **M1 + M2** Chuẩn bị dữ liệu + KNN đầu tiên so với mốc  
**Học trước (bài thư viện trên web, 3 chặng bắt buộc + checkpoint):** [Bài 8 Chuẩn bị feature và chia dữ liệu](https://vuongndlst.github.io/ml-level1/bai08/), [Bài 12 Học có giám sát](https://vuongndlst.github.io/ml-level1/bai12/), [Bài 13 KNN](https://vuongndlst.github.io/ml-level1/bai13/)


---
## Bước 0 — Nạp dữ liệu (đã viết sẵn, chỉ cần chạy)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
bo = load_breast_cancer(as_frame=True)          # có sẵn trong scikit-learn, không cần tải file
df = bo.frame.copy()
df["chan_doan"] = df["target"].map({0: "ác tính", 1: "lành tính"})
print(df.shape)
df.head()


### Câu hỏi của nhóm (M1)

*Gợi ý câu hỏi:* KNN đoán lành / ác đúng tới đâu? · Ca sai nguy hiểm hơn là loại nào? · Cần bao nhiêu hàng xóm?

| | Nhóm điền |
|---|---|
| Câu hỏi nhóm chọn | … |
| Cột sẽ dùng | … |
| Dự đoán trước (giả thuyết) | … |
| Làm sao biết dự đoán đúng hay sai? | … |


---
## Bước 1 — Khám phá

> **🎯 Mục tiêu:** Biết nhãn là gì, có bao nhiêu lớp, các lớp có cân bằng không.
>
> **🧭 Vì sao bước này?** Chưa hiểu mỗi dòng, mỗi cột là gì thì không biết hỏi dữ liệu câu gì, cũng không biết cột nào là đáp án (nhãn) và cột nào dùng để đoán (feature).
>
> **📋 Nhiệm vụ:**
> 1. Đếm số mẫu mỗi lớp.
> 2. Liệt kê các cột sẽ dùng làm feature.
> 3. Xem thang đo các cột: có cột nào lớn gấp nhiều lần cột khác không?
>
> **💡 Gợi ý lệnh:**
> - `df["..."].value_counts()` — đếm mỗi lớp
> - `df.describe()` — thang đo từng cột
>
> **✅ Đạt khi:** nói được nhãn, số lớp, các cột feature và cột nào có thang đo rất lớn.


In [ ]:
# Buoc 1 - code cua nhom



**Nhận xét của nhóm:** …


---
## Bước 2 — Chuẩn bị dữ liệu

> **🎯 Mục tiêu:** Chọn X, y; chia train / validation / test 60 / 20 / 20; đưa về 0 – 1 đúng cách.
>
> **🧭 Vì sao bước này?** Model chỉ được học trên tập huấn luyện. Tập validation dùng để chọn K; tập test giữ kín tới cuối — nếu nhìn test để chọn K thì con số cuối cùng sẽ đẹp giả. Thang đo cũng chỉ được học trên tập huấn luyện.
>
> **📋 Nhiệm vụ:**
> 1. Chọn cột feature X và nhãn y.
> 2. Chia 3 tập, `stratify` để mỗi tập giữ tỉ lệ các lớp.
> 3. `MinMaxScaler` chỉ `fit` trên tập huấn luyện, rồi biến đổi cả ba tập.
>
> **💡 Gợi ý lệnh:**
> - `train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)` — tách test 20%
> - `MinMaxScaler().fit(X_train)` — học thang đo trên train
>
> **✅ Đạt khi:** in được số mẫu ba tập, thang đo chỉ học trên train.


In [ ]:
# Buoc 2 - code cua nhom
X = df[list(bo.feature_names)]          # 30 số đo từ ảnh tế bào
y = df["chan_doan"]
# nhóm viết tiếp phần chia và đưa về 0 - 1


**Nhận xét của nhóm:** …


---
## Bước 3 — Mốc và KNN đầu tiên

> **🎯 Mục tiêu:** So KNN K = 5 với mốc “luôn đoán lớp nhiều nhất” trên validation.
>
> **🧭 Vì sao bước này?** Không có mốc thì không biết 95% là giỏi hay bình thường. Mốc đơn giản nhất: luôn đoán lớp nhiều nhất.
>
> **📋 Nhiệm vụ:**
> 1. Tạo mốc bằng `DummyClassifier`.
> 2. Huấn luyện KNN K = 5 trên dữ liệu đã đưa về 0 – 1.
> 3. So hai con số trên **validation**.
>
> **💡 Gợi ý lệnh:**
> - `DummyClassifier(strategy="most_frequent")` — mốc
> - `KNeighborsClassifier(n_neighbors=5)` — KNN
>
> **✅ Đạt khi:** có hai con số: mốc và KNN, cùng trên validation.


In [ ]:
# Buoc 3 - code cua nhom



**Nhận xét của nhóm:** …


### ✅ Nộp M1 + M2 (tuần 1)

- [ ] Bảng câu hỏi của nhóm đã điền
- [ ] Bước 1 – 3 có nhận xét kèm số
- [ ] *Runtime → Restart and run all* chạy hết không lỗi
- [ ] Link notebook (*Chia sẻ → Bất kỳ ai có đường liên kết · Người xem*) nộp lên Canvas trước 23:59 Thứ 7


---
# TUẦN 2 (buổi 12) — Chọn K, chấm test, kể kết quả

**Mốc của tuần:** **M3 + M4** Chọn K, chấm test một lần, 1 slide  
**Học trước (bài thư viện trên web, 3 chặng bắt buộc + checkpoint):** [Bài 13 KNN — chọn K, thang đo](https://vuongndlst.github.io/ml-level1/bai13/)


---
## Bước 4 — Chọn K

> **🎯 Mục tiêu:** Thử K = 1, 3, …, 25 trên validation, chọn K tốt nhất; kiểm tra vai trò của thang đo.
>
> **🧭 Vì sao bước này?** K nhỏ: nghe theo một hàng xóm, dễ bị nhiễu. K lớn: hỏi quá nhiều người ở xa. Chọn K bằng dữ liệu validation.
>
> **📋 Nhiệm vụ:**
> 1. Vẽ độ chính xác validation theo K.
> 2. Chọn K tốt nhất.
> 3. Cùng K đó nhưng **không** đưa về 0 – 1: kết quả?
>
> **💡 Gợi ý lệnh:**
> - `[... for k in ds_k]` — thử nhiều K bằng một dòng
> - `max(acc_val)` — độ chính xác cao nhất; hoà thì chọn K lớn hơn
>
> **✅ Đạt khi:** có biểu đồ K – độ chính xác, K đã chọn và so sánh có / không đưa về 0 – 1.


In [ ]:
# Buoc 4 - code cua nhom



**Nhận xét của nhóm:** …


---
## Bước 5 — Chấm test và đọc lỗi

> **🎯 Mục tiêu:** Chấm test đúng một lần, đọc ma trận nhầm lẫn và các ca sai.
>
> **🧭 Vì sao bước này?** Chấm test đúng một lần, sau khi đã chốt mọi lựa chọn. Ma trận nhầm lẫn cho biết model nhầm lớp nào với lớp nào.
>
> **📋 Nhiệm vụ:**
> 1. Huấn luyện KNN với K đã chọn, chấm trên test.
> 2. Vẽ ma trận nhầm lẫn: lớp nào bị nhầm với lớp nào?
> 3. Xem các ca sai — có điểm gì chung?
>
> **💡 Gợi ý lệnh:**
> - `confusion_matrix(y_test, du)` — ma trận nhầm lẫn
> - `X_test[du != y_test]` — các ca sai
>
> **✅ Đạt khi:** có độ chính xác test, ma trận nhầm lẫn, một nhận xét về ca sai.


In [ ]:
# Buoc 5 - code cua nhom



**Nhận xét của nhóm:** …


### Slide kết quả (M4) — một slide, 4 ô

| Ô | Nội dung |
|---|---|
| Câu hỏi | … |
| Một biểu đồ | … |
| Một con số | … |
| Giới hạn | … |

*Làm trên Canva hoặc Google Slides; dán link vào ô nộp Canvas cùng link notebook.*


---
## Bước 6 — Nhật ký AI

Không dùng AI thì ghi *Không dùng*. Kiểm chứng = nhóm tự chạy lại, đối chiếu số, sửa chỗ AI sai.

| Nhóm hỏi AI | AI trả lời / viết | Nhóm kiểm chứng và sửa |
|---|---|---|
| … | … | … |
| … | … | … |
| … | … | … |


---
## Mở rộng — nhóm xong sớm (không bắt buộc)

1. Chỉ dùng 3 cột `mean radius`, `mean texture`, `mean smoothness` — K tốt nhất và độ chính xác đổi thế nào?
2. Ca **ác tính bị đoán lành tính** nguy hiểm hơn ca ngược lại. Có cách nào giảm loại sai này không?


In [ ]:
# Mo rong - code cua nhom



---
## Trước khi nộp

- [ ] *Runtime → Restart and run all* chạy hết không lỗi
- [ ] Mỗi bước có một ô nhận xét **có số và cỡ nhóm**
- [ ] Không có câu “… **gây ra** …” khi dữ liệu chỉ cho thấy đi cùng nhau
- [ ] Bảng vai trò và nhật ký AI đã điền
- [ ] Link notebook ở chế độ *Người xem*, dán lên Canvas cùng link Canva
